# 01 — Inspect the raw dataset

Converted from `scripts/01_inspect.py` (Derrick, data prep).

First look at the full **FreshRetailNet-50K** dataset before cutting the working subset: shape, columns, types, a sample row, distinct counts, date range, and how many series are complete.

> Run this notebook from the `Data Prep - Derrick` folder. The first cell downloads the full dataset from the Hugging Face Hub (large, cached after the first run).

### Where this fits — CRISP-DM Phase 2: Data Understanding

This notebook is the **Data Understanding** phase of CRISP-DM (Week 5): *collect the
initial data, describe it, explore it, and verify its quality* before we build
anything.

**The dataset.** FreshRetailNet-50K (Dingdong-Inc, Hugging Face) is a fresh-grocery
retail dataset. The published size is **4.5 million training rows + 350,000
evaluation rows**. The "50K" in the name refers to about **50,000 store-product
series** (4,500,000 / 90 days = 50,000): each series is one product in one store
followed over 90 daily records. Every row carries scaled daily sales, 24-hour sales
and stock-status arrays, promotion/holiday flags, a discount ratio, stockout hours,
and weather.

**What the cells below do, in CRISP-DM terms:**

| CRISP-DM task (Phase 2) | Cell(s) in this notebook |
| --- | --- |
| Collect initial data | `load_dataset(...)` |
| Describe data | row / column counts, column types, one sample row |
| Explore data | distinct counts, date range, stores & series per city |
| Verify data quality | days-per-series, count of complete 90-day series |

**Levels of measurement (Week 6).** Classifying each variable tells us which
operations are valid — e.g. an id is a label, so averaging it is meaningless:

| Variable | Measurement level |
| --- | --- |
| `city_id`, `store_id`, `product_id`, `management_group_id`, the category ids | Nominal (categorical codes) |
| `holiday_flag`, `activity_flag` | Nominal / binary |
| `dt` | Interval (calendar date) |
| `avg_temperature` | Interval (0 is arbitrary) |
| `sale_amount`, `hours_sale`, `discount`, `stock_hour6_22_cnt`, `precpt`, `avg_humidity`, `avg_wind_level` | Ratio (true zero) |

This is why Notebook 02 stores the id columns as `category`, not numbers.

In [ ]:
# MIS 480 group project - look at the raw dataset before cutting our subset
# Derrick, data prep
import pandas as pd
from datasets import load_dataset

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

In [ ]:
ds = load_dataset("Dingdong-Inc/FreshRetailNet-50K", split="train")

In [ ]:
print(f"rows: {ds.num_rows}")
print(f"columns: {ds.column_names}")

In [ ]:
print("\ncolumn types:")
for name, feat in ds.features.items():
    print(f"  {name}: {feat}")

In [ ]:
print("\nsample row:")
for k, v in ds[0].items():
    show = v if not isinstance(v, list) else f"list(len={len(v)}): {v[:6]}..."
    print(f"  {k}: {show}")

In [ ]:
print("\ndistinct counts:")
for col in ["city_id", "store_id", "product_id", "dt"]:
    print(f"  {col}: {len(ds.unique(col))}")

In [ ]:
idx = ds.select_columns(["city_id", "store_id", "product_id", "dt"]).to_pandas()
idx["dt"] = pd.to_datetime(idx["dt"])

In [ ]:
print(f"\ndate range: {idx['dt'].min().date()} to {idx['dt'].max().date()} ({idx['dt'].nunique()} days)")

In [ ]:
print("\nstores per city (top 10):")
print(idx.groupby("city_id")["store_id"].nunique().sort_values(ascending=False).head(10).to_string())

In [ ]:
print("\nseries per city (top 10):")
print(idx.groupby("city_id").apply(lambda g: g.groupby(["store_id","product_id"]).ngroups).sort_values(ascending=False).head(10).to_string())

In [ ]:
days = idx.groupby(["store_id","product_id"])["dt"].nunique()
print(f"\ndays per series: min {days.min()}, max {days.max()}")
print(f"complete (90-day) series: {(days == 90).sum()} of {len(days)}")

---
**Pipeline:** **01 inspect** → 02 subset → 03 EDA → 04 modeling → 05 classification.

**Phase 2 outcome.** The data is well-formed (0 missing, complete 90-day series) and we understand the variables and their measurement levels. This hands off to **Phase 3 (notebook 02)**, where we select the working subset and engineer features.